# CLRerNet lane export for the iRAP screening pipeline

Runs CLRerNet on dashcam frames in Colab and writes **one JSON dump** that the
local pipeline converts into canonical detections:

```bash
uv run irap import-lanes --route ROUTE_DIR --raw clrernet_raw.json
```

**This notebook does no coordinate arithmetic.** It serialises CLRerNet's raw
normalised points, its confidences, and the geometry this run actually used.
Every transform that could be plausibly wrong lives in
`irap_screening.runners.lane`, where it is unit tested. The mmcv-pinned
environment below is not reproducible, so no arithmetic should depend on it.

| Block | Purpose |
|-------|---------|
| 0 | Install everything. **One time only — restart the runtime afterwards.** |
| 1 | Patch CLRerNet and mmdet for inference-only use |
| 2 | Verify the dependency versions |
| 3 | Mount Drive, set paths, fetch weights |
| 4 | Load the model and set its output geometry |
| 5 | Inference helpers |
| 6 | Run inference over every frame |
| 7 | **Write `clrernet_raw.json`** |
| 8 | Sanity overlay — confirm the points land on the markings |
| 9 | Optional: write a `frames.csv` skeleton for the route |

## Order matters

Block 4 derives `cut_height` from `CROP_TOP`. Those two must agree: CLRerNet
maps its output onto the rows it believes were cut, so a mismatch stretches
every lane vertically and still produces lane-shaped output. The local exporter
rejects a dump where they disagree.


In [ ]:
# ===== Block 0 - install. ONE TIME ONLY. Restart the runtime afterwards. =====
import os
import subprocess


def run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[FAIL] {' '.join(cmd)}")
        print(result.stderr[-2000:])
        raise SystemExit(1)


print("Installing PyTorch 2.2.0 ...")
run(["pip", "install", "torch==2.2.0", "torchvision==0.17.0", "torchaudio==2.2.0",
     "--index-url", "https://download.pytorch.org/whl/cu121", "--quiet"])

print("Installing numpy 1.26.4 ...")
run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "--quiet"])

print("Installing opencv ...")
run(["pip", "install", "opencv-python==4.9.0.80", "--quiet"])

print("Installing mmengine ...")
run(["pip", "install", "openmim", "mmengine", "--quiet"])

print("Installing mmcv 2.2.0 ...")
run(["pip", "install", "mmcv==2.2.0", "-f",
     "https://download.openmmlab.com/mmcv/dist/cu121/torch2.2.0/index.html", "--quiet"])

print("Installing mmdet ...")
run(["pip", "install", "mmdet", "--quiet"])
run(["pip", "install", "yapf==0.40.1", "tensorboard", "--quiet"])

if not os.path.exists("/content/CLRerNet"):
    print("Cloning CLRerNet ...")
    run(["git", "clone", "https://github.com/hirotomusiker/CLRerNet.git", "/content/CLRerNet"])

print("Compiling the nms extension ...")
run(["pip", "install", "setuptools==68.2.2", "--quiet"])
result = subprocess.run(["python", "setup.py", "install"], capture_output=True, text=True,
                        cwd="/content/CLRerNet/libs/models/layers/nms")
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise SystemExit("nms compilation failed")

print("\nAll packages installed.")
print("NOW: Runtime > Restart session, then continue from Block 1.")


---
**Restart the runtime now** (Runtime > Restart session), then run Block 1.
Skipping the restart leaves the old numpy loaded and mmcv will fail to import.

---


In [ ]:
# ===== Block 1 - patches for inference-only use =====
# CLRerNet's repo expects its training environment. These patches remove the
# pieces that only matter for training or evaluation.
#
# mmdet is located WITHOUT importing it. Its __init__ carries the very mmcv
# version assert patch 1 removes, and importing it also runs the transformers
# chain patch 4 repairs -- so `import mmdet` here would fail on exactly the
# problems this cell exists to fix. find_spec reads the filesystem and never
# executes the module.
import importlib.util
import subprocess
from pathlib import Path

spec = importlib.util.find_spec("mmdet")
if spec is None or not spec.submodule_search_locations:
    raise SystemExit(
        "mmdet is not installed. Run Block 0, then restart the runtime "
        "(Runtime > Restart session) before running this cell."
    )
MMDET_DIR = Path(spec.submodule_search_locations[0])
print(f"mmdet at {MMDET_DIR}")

CLRERNET = Path("/content/CLRerNet")
if not CLRERNET.exists():
    raise SystemExit(f"{CLRERNET} is missing. Run Block 0 first.")

# 1. mmdet asserts a narrow mmcv version range; we run a newer mmcv on purpose.
init_path = MMDET_DIR / "__init__.py"
text = init_path.read_text()
needle = "assert (mmcv_version >= digit_version(mmcv_minimum_version)"
if needle in text:
    init_path.write_text(text.replace(
        needle, "assert (True or mmcv_version >= digit_version(mmcv_minimum_version)"))
    print("mmdet version check patched.")
else:
    print("[SKIP] mmdet version check already patched or restructured.")

# 2. The CULane metric is evaluation-only and pulls in extra build steps.
ds_init = CLRERNET / "libs/datasets/__init__.py"
text = ds_init.read_text()
ds_init.write_text(text.replace(
    "from .metrics.culane_metric import CULaneMetric  # noqa: F401",
    "# from .metrics.culane_metric import CULaneMetric  # disabled for inference"))
print("culane_metric import patched.")

# 3. The DLA backbone downloads ImageNet weights it does not need: the checkpoint
#    carries them.
dla = CLRERNET / "libs/models/backbones/dla.py"
text = dla.read_text()
old = """    def load_pretrained_model(self, data='imagenet', name='dla34', hash='ba72cf86'):
        # fc = self.fc
        if name.endswith('.pth'):
            model_weights = torch.load(data + name)
        else:
            model_url = get_model_url(data, name, hash)
            model_weights = model_zoo.load_url(model_url)
        self.load_state_dict(model_weights, strict=False)
        # self.fc = fc"""
new = """    def load_pretrained_model(self, data='imagenet', name='dla34', hash='ba72cf86'):
        pass  # disabled for inference-only mode"""
if old in text:
    text = text.replace(old, new)
    print("load_pretrained_model patched.")
else:
    print("[SKIP] load_pretrained_model exact match failed.")
text = text.replace("            pretrained=pretrained, levels=levels, in_channels=in_channels",
                    "            pretrained=False, levels=levels, in_channels=in_channels")
dla.write_text(text)
print("DLA pretrained=False patched.")

# 4. mmdet imports transformers for GroundingDINO fusion, which breaks on
#    torch < 2.4. CLRerNet uses DLANet, so failing that import is harmless.
vlfuse = MMDET_DIR / "models" / "utils" / "vlfuse_helper.py"
if vlfuse.exists():
    text = vlfuse.read_text()
    patched = text.replace("except ImportError:",
                           "except (ImportError, NameError, AttributeError):")
    if patched != text:
        vlfuse.write_text(patched)
        print("vlfuse_helper exception widened.")
    else:
        print("vlfuse_helper already patched.")

subprocess.run(["find", "/content/CLRerNet", "-name", "*.pyc", "-delete"])
subprocess.run(["find", "/content/CLRerNet", "-name", "__pycache__", "-type", "d",
                "-exec", "rm", "-rf", "{}", "+"], capture_output=True)
print("\nBlock 1 complete.")


In [ ]:
# ===== Block 2 - dependency verification =====
import sys

import cv2
import mmcv
import mmdet
import mmengine
import numpy as np
import torch

sys.path.insert(0, "/content/CLRerNet")
sys.path.insert(0, "/content/CLRerNet/libs")

from mmdet.apis import init_detector  # noqa: E402

assert torch.__version__.startswith("2.2"), f"torch is {torch.__version__}, expected 2.2.x"
assert int(np.__version__.split(".")[0]) < 2, f"numpy is {np.__version__}, expected 1.x"
assert mmcv.__version__ == "2.2.0", f"mmcv is {mmcv.__version__}, expected 2.2.0"

print("Dependency check passed.")
print(f"  torch    : {torch.__version__}")
print(f"  numpy    : {np.__version__}")
print(f"  opencv   : {cv2.__version__}")
print(f"  mmengine : {mmengine.__version__}")
print(f"  mmcv     : {mmcv.__version__}")
print(f"  mmdet    : {mmdet.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")


In [ ]:
# ===== Block 3 - paths, images, weights =====
import shutil
import urllib.request
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# ---------------------------- USER-EDITABLE ----------------------------------
IMAGE_DIR = Path("/content/drive/MyDrive/fyrp - dataset demo/frames2")
OUTPUT_DIR = IMAGE_DIR / "output"
MODEL_DIR = IMAGE_DIR.parent / "models"
SAMPLING_INTERVAL = 1          # take every Nth image
# -----------------------------------------------------------------------------

WEIGHTS_NAME = "clrernet_culane_dla34_ema.pth"
WEIGHTS_URL = ("https://github.com/hirotomusiker/CLRerNet/releases/download/"
               f"v0.1.0/{WEIGHTS_NAME}")
WEIGHTS_DRIVE = MODEL_DIR / WEIGHTS_NAME
WEIGHTS_LOCAL = Path("/content/CLRerNet") / WEIGHTS_NAME
CONFIG_PATH = Path("/content/CLRerNet/configs/clrernet/culane/clrernet_culane_dla34_ema.py")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FORMATS = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"]
all_images = sorted({str(p) for fmt in IMAGE_FORMATS for p in IMAGE_DIR.glob(fmt)})
assert all_images, f"No images found in {IMAGE_DIR}"
sampled_images = all_images[::SAMPLING_INTERVAL]
print(f"{len(all_images)} images found, {len(sampled_images)} sampled.")

if WEIGHTS_DRIVE.exists():
    shutil.copy(WEIGHTS_DRIVE, WEIGHTS_LOCAL)
    print(f"Weights loaded from Drive ({WEIGHTS_LOCAL.stat().st_size / 1e6:.1f} MB).")
else:
    print("Weights not on Drive, downloading (~63 MB) ...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_LOCAL)
    shutil.copy(WEIGHTS_LOCAL, WEIGHTS_DRIVE)
    print("Downloaded and cached to Drive.")

assert CONFIG_PATH.exists(), f"Config not found: {CONFIG_PATH}"
print("Block 3 complete.")


In [ ]:
# ===== Block 4 - load the model and fix its output geometry =====
import os

import torch

# ---------------------------- USER-EDITABLE ----------------------------------
CROP_TOP = 0.25        # fraction of image height removed from the top
CROP_BOTTOM = 0.0     # fraction removed from the bottom (bonnet, dashboard)
MODEL_W, MODEL_H = 800, 320    # network input size the crop is resized to
CONF_THRESHOLD = 0.10  # permissive on purpose; see the note below
ORI_IMG_W, ORI_IMG_H = 1241, 376   # CLRerNet's reference geometry
# -----------------------------------------------------------------------------

# cut_height is DERIVED, never typed in. CLRerNet spreads its output over the
# rows it believes were kept, so cut_height/ORI_IMG_H must equal CROP_TOP. Typing
# them separately is how they drift, and the result is lanes stretched against
# the image that still look like lanes.
CUT_HEIGHT = int(round(ORI_IMG_H * CROP_TOP))

# CONF_THRESHOLD must stay at or below the pipeline's lane.clrernet_conf (0.41).
# The dump's floor is baked into the cached artifact: anything dropped here can
# never be reconsidered locally, and a sensitivity sweep below this floor would
# report no change while in fact changing nothing. irap import-lanes rejects a
# dump that violates this.

os.makedirs("dataset/culane/list", exist_ok=True)
open("dataset/culane/list/test.txt", "a").close()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = init_detector(CONFIG_PATH, str(WEIGHTS_LOCAL), device=device)

model.bbox_head.test_cfg.conf_threshold = CONF_THRESHOLD
model.bbox_head.test_cfg.ori_img_h = ORI_IMG_H
model.bbox_head.test_cfg.ori_img_w = ORI_IMG_W
model.bbox_head.test_cfg.cut_height = CUT_HEIGHT

print(f"Model loaded on {device}.")
print(f"  conf_threshold : {CONF_THRESHOLD}")
print(f"  ori_img (w, h) : ({ORI_IMG_W}, {ORI_IMG_H})")
print(f"  cut_height     : {CUT_HEIGHT}  (= {ORI_IMG_H} * {CROP_TOP})")
print(f"  crop           : top {CROP_TOP:.0%}, bottom {CROP_BOTTOM:.0%}")


In [ ]:
# ===== Block 5 - inference helpers =====
import cv2
import numpy as np
import torch
from mmdet.structures import DetDataSample


def load_bgr(img_path):
    """Load any image as 3-channel BGR. CLRerNet was trained on colour CULane."""
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    assert img is not None, f"Failed to read image: {img_path}"
    return img


def run_clrernet(model, img_bgr):
    """Crop, resize, normalise and run one frame through CLRerNet."""
    h, w = img_bgr.shape[:2]
    crop_top_px = int(h * CROP_TOP)
    crop_bottom_px = int(h * CROP_BOTTOM)
    crop_end = h - crop_bottom_px if crop_bottom_px > 0 else h

    cropped = img_bgr[crop_top_px:crop_end, 0:w]
    resized = cv2.resize(cropped, (MODEL_W, MODEL_H), interpolation=cv2.INTER_LINEAR)

    mean = np.array([123.675, 116.28, 103.53], dtype=np.float32)
    std = np.array([58.395, 57.12, 57.375], dtype=np.float32)
    normalised = (resized.astype(np.float32) - mean) / std

    tensor = torch.from_numpy(normalised).permute(2, 0, 1).unsqueeze(0)
    tensor = tensor.to(next(model.parameters()).device)

    sample = DetDataSample()
    sample.set_metainfo({
        "img_shape": (MODEL_H, MODEL_W),
        "ori_shape": (MODEL_H, MODEL_W),
        "scale_factor": 1.0,
    })
    with torch.no_grad():
        return model.predict(tensor, [sample])[0]


# Quick check on one frame before committing to the whole run.
probe = load_bgr(sampled_images[len(sampled_images) // 2])
probe_result = run_clrernet(model, probe)
probe_scores = probe_result.get("scores", [])
if hasattr(probe_scores, "cpu"):
    probe_scores = probe_scores.cpu().tolist()
print(f"probe frame {probe.shape[1]}x{probe.shape[0]}: "
      f"{len(probe_result.get('lanes', []))} lanes, "
      f"scores {[round(float(s), 3) for s in probe_scores]}")


In [ ]:
# ===== Block 6 - run inference over every sampled frame =====
frame_results = []

for index, img_path in enumerate(sampled_images):
    img_bgr = load_bgr(img_path)
    result = run_clrernet(model, img_bgr)

    scores = result.get("scores", [])
    if hasattr(scores, "cpu"):
        scores = scores.cpu().tolist()

    frame_results.append({
        "filename": os.path.basename(img_path),
        "source_wh": [int(img_bgr.shape[1]), int(img_bgr.shape[0])],
        "lanes": result.get("lanes", []) or [],
        "scores": [float(s) for s in scores],
    })

    if (index + 1) % 10 == 0 or index == len(sampled_images) - 1:
        print(f"  [{index + 1}/{len(sampled_images)}] {frame_results[-1]['filename']} "
              f"- {len(frame_results[-1]['lanes'])} lanes")

detected = sum(1 for entry in frame_results if entry["lanes"])
print(f"\nInference complete: {len(frame_results)} frames, "
      f"{detected} with at least one lane ({detected / len(frame_results):.1%}).")


In [ ]:
# ===== Block 7 - write clrernet_raw.json =====
# This cell only writes numbers down. No coordinate arithmetic happens here or
# anywhere else in this notebook.
import json

RAW_PATH = OUTPUT_DIR / "clrernet_raw.json"

frames_payload = []
for entry in frame_results:
    lanes, scores = entry["lanes"], entry["scores"]
    # strict pairing: lanes and scores are parallel arrays from one result. A
    # length mismatch would mis-pair confidences, which the local lane module
    # filters on, and nothing downstream could detect it. Fail here instead.
    assert len(lanes) == len(scores), (
        f"{entry['filename']}: {len(lanes)} lanes but {len(scores)} scores")

    frames_payload.append({
        "filename": entry["filename"],
        "source_wh": entry["source_wh"],
        "lanes": [
            {
                "conf": float(score),
                "points_norm": [[float(x), float(y)]
                                for x, y in (lane.points if hasattr(lane, "points") else lane)],
            }
            for lane, score in zip(lanes, scores, strict=True)
        ],
    })

payload = {
    "meta": {
        "crop_top": float(CROP_TOP),
        "crop_bottom": float(CROP_BOTTOM),
        "cut_height": int(CUT_HEIGHT),
        "ori_img_h": int(ORI_IMG_H),
        "ori_img_w": int(ORI_IMG_W),
        "conf_threshold": float(CONF_THRESHOLD),
        "model_wh": [int(MODEL_W), int(MODEL_H)],
        "model": "clrernet_culane_dla34_ema",
        "sampling_interval": int(SAMPLING_INTERVAL),
    },
    "frames": frames_payload,
}

RAW_PATH.write_text(json.dumps(payload), encoding="utf-8")
total_lanes = sum(len(f["lanes"]) for f in frames_payload)
print(f"Wrote {RAW_PATH}")
print(f"  {len(frames_payload)} frames, {total_lanes} lanes, "
      f"{RAW_PATH.stat().st_size / 1024:.1f} KB")
assert total_lanes > 0, (
    "no lanes on any frame; the local exporter rejects such a dump because zero "
    "lanes grades every segment as the worst contrast band")


In [ ]:
# ===== Block 8 - sanity overlay =====
# Reproduces the inverse mapping the local exporter applies and draws it on the
# original frame. The polylines must sit on the lane markings. This is the check
# that distinguishes the correct reading of CLRerNet's normalised y from the
# competing one, which differs by a factor of ~0.73 and still looks like lanes.
import matplotlib.pyplot as plt

PREVIEW_INDEX = len(frame_results) // 2   # USER-EDITABLE

entry = frames_payload[PREVIEW_INDEX]
img = load_bgr(IMAGE_DIR / entry["filename"])
width, height = entry["source_wh"]

crop_top_px = int(height * CROP_TOP)
crop_bottom_px = int(height * CROP_BOTTOM)
crop_end = height - crop_bottom_px if crop_bottom_px > 0 else height
canvas_h = crop_end - crop_top_px
cut_ratio = CUT_HEIGHT / ORI_IMG_H

overlay = img.copy()
for lane in entry["lanes"]:
    points = [
        (x * width, crop_top_px + (y - cut_ratio) / (1.0 - cut_ratio) * canvas_h)
        for x, y in lane["points_norm"]
    ]
    for (x0, y0), (x1, y1) in zip(points, points[1:], strict=False):
        if not (0 <= x0 < width and 0 <= x1 < width):
            continue   # CLRerNet extends curved lanes past the frame edge
        cv2.line(overlay, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 3)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title(f"{entry['filename']} - {len(entry['lanes'])} lanes, mapped to original pixels")
plt.axis("off")
plt.show()
print("The green polylines must lie on the markings. If they are compressed "
      "towards the top of the road, cut_height and CROP_TOP disagree.")


In [ ]:
# ===== Block 9 (optional) - frames.csv skeleton =====
# irap import-lanes matches the dump to a route by image basename, so the route
# directory needs a frames.csv covering exactly these frames. This writes one
# with synthetic timestamps, which is enough to exercise the lane path. A real
# route's frames.csv should carry the capture timestamps, and needs gps.csv and
# imu.csv beside it before `irap report` will run.
import csv
import hashlib
from datetime import UTC, datetime, timedelta

FRAME_INTERVAL_S = 1.0    # USER-EDITABLE: seconds between sampled frames
START_TIME = datetime(2026, 1, 1, tzinfo=UTC)

CSV_PATH = OUTPUT_DIR / "frames.csv"
with CSV_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["frame_id", "path", "timestamp_utc", "sha256"])
    for index, entry in enumerate(frames_payload):
        digest = hashlib.sha256((IMAGE_DIR / entry["filename"]).read_bytes()).hexdigest()[:16]
        writer.writerow([
            f"f_{index:06d}",
            f"frames/{entry['filename']}",
            (START_TIME + timedelta(seconds=index * FRAME_INTERVAL_S)).isoformat(),
            digest,
        ])

print(f"Wrote {CSV_PATH} with {len(frames_payload)} rows.")
print("Copy the images into ROUTE_DIR/frames/ alongside it, then run locally:")
print("  uv run irap import-lanes --route ROUTE_DIR --raw clrernet_raw.json")


## Next

1. Download `clrernet_raw.json` (and `frames.csv`, if Block 9 was used) from
   `OUTPUT_DIR` on Drive.
2. Assemble the route directory locally:

   ```
   ROUTE_DIR/
     frames.csv
     frames/          <- the same images this notebook read
     gps.csv          <- required by irap report, not by import-lanes
     imu.csv          <- required by irap report, not by import-lanes
   ```

3. Convert:

   ```bash
   uv run irap import-lanes --route ROUTE_DIR --raw clrernet_raw.json
   ```

   This writes `ROUTE_DIR/detections/lane.jsonl`. It fails loudly on a dump
   whose geometry is inconsistent, whose frames do not match `frames.csv` in
   both directions, whose confidence floor sits above `lane.clrernet_conf`, or
   which found no lanes at all.

4. With `gps.csv` and `imu.csv` in place, run the full pipeline:

   ```bash
   uv run irap report --route ROUTE_DIR --out out
   ```

The dump is worth keeping. Every threshold sweep in the CPU layer runs against
cached detections, so this notebook is a one-time cost per route.
